In [1]:
# Import some useful modules.
import jax
import jax.numpy as jnp
import os

# Import JAX-FEM specific modules.
from jax_fem.problem import Problem
from jax_fem.solver import solver
from jax_fem.utils import save_sol
from jax_fem.generate_mesh import box_mesh_gmsh, get_meshio_cell_type, Mesh
import jax.random as jr 
jax.config.update("jax_enable_x64", True)

from core.utils import *
from core.model import SparseHyperelasticityGP
from core.material_models import get_material
from core.datasetclass import BenchmarkDataset, TractionDataset

       __       ___      ___   ___                _______  _______ .___  ___. 
      |  |     /   \     \  \ /  /               |   ____||   ____||   \/   | 
      |  |    /  ^  \     \  V  /      ______    |  |__   |  |__   |  \  /  | 
.--.  |  |   /  /_\  \     >   <      |______|   |   __|  |   __|  |  |\/|  | 
|  `--'  |  /  _____  \   /  .  \                |  |     |  |____ |  |  |  | 
 \______/  /__/     \__\ /__/ \__\               |__|     |_______||__|  |__| 
                                                                              



[01-25 11:50:04][INFO] jax_fem: pyamgx not installed. AMGX solver disabled.


In [2]:
model_path = "/home/mmdiscovery/shared/saved_model/20260104T185757/" # Replace with the actual path to your saved model
with open(os.path.join(model_path, "best_params.npy"), "rb") as f:
    best_params = jnp.load(f, allow_pickle=True).item()

with open(os.path.join(model_path, "z_stacked.npy"), "rb") as f:
    Z_stacked = jnp.load(f, allow_pickle=True)
learned_gp = SparseHyperelasticityGP(best_params, Z_stacked)

In [3]:
learned_gp.params

{'lengthscales': Array([ 4.06428, 10.23406], dtype=float64),
 'sigma_scaling': Array(8.82044, dtype=float64),
 'sigma_poly': Array(2.71828, dtype=float64),
 'offset': Array(1., dtype=float64),
 'growth_constant': Array(1.50873, dtype=float64),
 'poly_degree': Array(2., dtype=float64),
 'inducing_latent_variable_mean': Array([-1.09381,  3.5781 ,  1.62879,  1.92812,  2.80069,  0.07502,  1.02301,  2.30195,  2.38588,  0.53798,  0.20343,  1.78152,  0.77528,  2.12769,  1.25779, -0.94533,  2.32997,  1.01844,  0.86877,  3.26405,  1.42008, -0.12166,  0.20178,  0.60433, -0.2205 ], dtype=float64),
 'inducing_latent_variable_var': Array([0.00002, 0.00001, 0.00029, 0.00013, 0.00189, 0.00002, 0.00007, 0.00059, 0.00023, 0.00005, 0.00002, 0.00021, 0.00004, 0.00016, 0.00013, 0.00006, 0.0003 , 0.00001, 0.00003, 0.0002 , 0.00002, 0.00011, 0.00006, 0.00002, 0.00001], dtype=float64)}

In [ ]:
# base_save_path = "saved_model"  # change as needed
# os.makedirs(base_save_path, exist_ok=True)

# # Subfolder with datetime
# timestamp = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
# save_path = os.path.join(base_save_path, timestamp)
# os.makedirs(save_path, exist_ok=True)
material_model = "isihara"
dataset = TractionDataset("dataset",material_model)
F_all = []
u_cells_list = []
for loadstep in range(len(dataset)) :

    data = dataset[loadstep]
    coords = data["mesh_pos"][:,:2]
    cells = data["cells"]
    # u = data["u"]
    percent_noise = 0.0000
    node_type = data["node_type"]
    ux = data["u"][:, 0]
    ux[(data["node_type"] != 1)] += np.random.normal(0, percent_noise * 1, ux.shape)[(data["node_type"] != 1)]
    uy = data["u"][:, 1]
    uy[(data["node_type"] != 2)] += np.random.normal(0, percent_noise * 1, uy.shape)[(data["node_type"] != 2)]

    # Combine components into the full displacement vector u
    u = np.column_stack((ux, uy))
    # u[node_type == 0] = u[node_type == 0] + jax.random.normal(jr.key(0), u.shape)[node_type == 0] * 0.01 * mean_u
    load_parameter = data["load_parameter"]

    coord_cells = coords[cells]
    u_cells = u[cells]

    F, dNdx = deformation_gradient_element(coord_cells, u_cells)
    u_cells_list.append(u_cells)
    F_all.append(F)
# I_obs_all,_ = invariants_and_derivatives_vmap(F_all_stacked)
I_obs, _ = jax.vmap(invariants_and_derivatives)(F)

In [5]:
psi_gp = jax.vmap(learned_gp.psi)(f)

In [7]:
psi_gp

Array([ 1.8856 ,  1.88329,  1.88112,  1.88171,  1.88975,  1.88792,  1.88357,  1.8818 ,  1.88122,  1.88129,  1.89317,  1.89028,  1.88706,  1.88373,  1.88223,  1.88408,  1.88246,  1.88141,  1.88009,  1.8822 ,  1.89543,  1.8929 ,  1.88619,  1.88584,  1.88558,  1.88254,  1.88512,  1.88419,  1.88323,  1.8878 ,  1.88513,  1.8849 ,  1.88122,  1.88023,  1.89633,  1.89274,  1.88669,  1.8834 ,  1.88498,  1.88151,  1.87972,  1.88372,  1.88645,  1.88377,  1.8856 ,  1.88451,  1.88578,  1.88022,  1.88352,  1.88178,  1.89685,  1.89441,  1.89042,  1.88792,  1.88291,  1.88382,  1.88057,  1.87958,  1.88051,  1.88345,  1.88769,  1.88717,  1.88762,  1.88512,  1.88455,  1.88594,  1.88023,  1.8823 ,  1.88264,  1.88163,  1.90087,  1.89194,  1.89002,  1.88855,  1.88701,  1.88245,  1.88354,  1.88065,  1.87805,  1.87994,  1.88058,  1.88344,  1.88795,  1.89236,  1.88933,  1.8861 ,  1.88436,  1.88459,  1.88561,  1.87813,  1.88239,  1.88139,  1.88252,  1.88129,  1.89833,  1.89121,  1.88782,  1.88826,  1.88681,
   

In [2]:
from core.material_models import BaseMaterialModel

In [ ]:
from core.material_models import BaseMaterialModel
class MooneyRivlin(BaseMaterialModel):
    def __init__(self, params, jit_P: bool = True):
        super().__init__(jit_P=jit_P)
        self.params = params
        self.dev_params = self.params[:-1]
        self.vol_param = self.params[-1]

    def phi(self, f) -> jnp.ndarray:
        print(f.shape)
        I,_ = invariants_and_derivatives(f)
        i3 = I[2] + 1e-6
        i1_dev = i3 ** (-1/3) * I[0]
        i2_dev = i3 ** (-2/3) *I[1]

        X = i1_dev - 3.0
        Y = i2_dev - 3.0
        
        # --- Deviatoric Terms (W) ---
        # Assuming dev_params = [c01, c02, c10, c11, c12, c20, c21, c22]
        # Using the standard N=2 Polynomial Model terms (C10, C01, C20, C11, C02)
        dev_terms = (
            # C10 * X
            self.dev_params[0] * X + 
            # C01 * Y
            self.dev_params[1] * Y + 
            # C20 * X**2
            self.dev_params[2] * X**2 +
            # C11 * X * Y
            self.dev_params[3] * X * Y + 
            # C02 * Y**2
            self.dev_params[4] * Y**2 

            # self.dev_params[5] * X*Y**2 + 

            # self.dev_params[6] * X**2 * Y + 

            # self.dev_params[7] * X**2 * Y ** 2

            # Add C12, C21, C22 terms here if required by your specific model definition
        )
        
        # --- Volumetric Terms (U) ---
        # Assuming vol_params = [d0, d1] are D2 and D1 parameters (inverse bulk moduli)
        J = jnp.sqrt(i3)
        J_minus_1 = J - 1.0

        # Assuming the volumetric function U(J) = (1/D1)(J-1)^2 + (1/D2)(J-1)^4
        # with D1=d1 and D2=d0 (or vice versa, depending on convention)
        
        # D1 is typically the lower order term (quadratic, hence d1)
        # D2 is typically the higher order term (quartic, hence d0)
        vol_terms = (
            # (1/D1) * (J - 1)**2
            (self.vol_param) * J_minus_1**2 
        #     # (1/D2) * (J - 1)**4
        #     (self.vol_params[1]) * J_minus_1**4
        )
        
        return dev_terms + vol_terms

In [34]:
f.shape

(2688, 3, 3)

In [35]:
def loss(psi_gp, params, f) :
    mr = MooneyRivlin(jnp.exp(params))
    psi_true = jax.vmap(mr.phi)(f)
    return jnp.sum((psi_gp - psi_true)**2)

In [36]:
import optax

# trainable parameters
params = jnp.zeros(9) 

# choose optimizer
# opt = optax.lbfgs(1e-3)
opt = optax.adam(5e-3)
opt_state = opt.init(params)
# JIT the loss and gradients
loss_and_grad = jax.jit(jax.value_and_grad(
    lambda params: loss(psi_gp, params, f)
))


for step in range(50000*3):
    loss, grads = loss_and_grad(params)
    updates, opt_state = opt.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

    if step % 50 == 0:
        print(f"step {step:04d}  loss={loss:.6f} {jnp.exp(params)}")



(3, 3)
step 0000  loss=71.723639 [1.00501 1.00501 1.00501 1.      1.      1.      1.      1.      1.00501]
step 0050  loss=5.756364 [1.1513  1.16848 0.94344 1.      1.      1.      1.      1.      1.19642]
step 0100  loss=1.694536 [1.08569 1.13723 0.76543 1.      1.      1.      1.      1.      1.23431]
step 0150  loss=0.772778 [1.04537 1.11565 0.69883 1.      1.      1.      1.      1.      1.262  ]
step 0200  loss=0.540076 [1.01885 1.0985  0.67453 1.      1.      1.      1.      1.      1.27817]
step 0250  loss=0.466052 [1.00106 1.08507 0.67159 1.      1.      1.      1.      1.      1.28806]
step 0300  loss=0.421980 [0.98756 1.07355 0.67956 1.      1.      1.      1.      1.      1.29488]
step 0350  loss=0.382145 [0.97566 1.06267 0.69274 1.      1.      1.      1.      1.      1.30048]
step 0400  loss=0.343310 [0.96409 1.05181 0.70815 1.      1.      1.      1.      1.      1.30573]
step 0450  loss=0.305802 [0.95244 1.04081 0.72441 1.      1.      1.      1.      1.      1.31097]
st

KeyboardInterrupt: 

In [ ]:
# base_save_path = "saved_model"  # change as needed
# os.makedirs(base_save_path, exist_ok=True)

# # Subfolder with datetime
# timestamp = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
# save_path = os.path.join(base_save_path, timestamp)
# os.makedirs(save_path, exist_ok=True)
material_model = "mooneyrivlin"
dataset = TractionDataset("dataset",material_model)
F_all = []
for loadstep in range(len(dataset)) :

    data = dataset[loadstep]
    coords = data["mesh_pos"][:,:2]
    cells = data["cells"]
    # u = data["u"]
    percent_noise = 0.00
    node_type = data["node_type"]
    ux = data["u"][:, 0]
    ux[(data["node_type"] != 1)] += np.random.normal(0, percent_noise * 1, ux.shape)[(data["node_type"] != 1)]
    uy = data["u"][:, 1]
    uy[(data["node_type"] != 2)] += np.random.normal(0, percent_noise * 1, uy.shape)[(data["node_type"] != 2)]

    # Combine components into the full displacement vector u
    u = np.column_stack((ux, uy))
    # u[node_type == 0] = u[node_type == 0] + jax.random.normal(jr.key(0), u.shape)[node_type == 0] * 0.01 * mean_u
    load_parameter = data["load_parameter"]

    coord_cells = coords[cells]
    u_cells = u[cells]

    F, dNdx = deformation_gradient_element(coord_cells, u_cells)
    F_all.append(F)

# I_obs_all,_ = invariants_and_derivatives_vmap(F_all_stacked)
I_obs, _ = jax.vmap(invariants_and_derivatives)(F)

In [2]:
from core.material_models import BaseMaterialModel
class MooneyRivlin(BaseMaterialModel):
    def __init__(self, params, jit_P: bool = True):
        super().__init__(jit_P=jit_P)
        self.params = params
        self.dev_params = self.params[:-1]
        self.vol_param = self.params[-1]

    def phi(self, f) -> jnp.ndarray:
        print(f.shape)
        I,_ = invariants_and_derivatives(f)
        i3 = I[2] + 1e-6
        i1_dev = i3 ** (-1/3) * I[0]
        i2_dev = i3 ** (-2/3) *I[1]

        X = i1_dev - 3.0
        Y = i2_dev - 3.0
        
        # --- Deviatoric Terms (W) ---
        # Assuming dev_params = [c01, c02, c10, c11, c12, c20, c21, c22]
        # Using the standard N=2 Polynomial Model terms (C10, C01, C20, C11, C02)
        dev_terms = (
            # C10 * X
            self.dev_params[0] * X + 
            # C01 * Y
            self.dev_params[1] * Y + 
            # C20 * X**2
            self.dev_params[2] * X**2 +
            # C11 * X * Y
            self.dev_params[3] * X * Y + 
            # C02 * Y**2
            self.dev_params[4] * Y**2  

            # self.dev_params[5] * X*Y**2 + 

            # self.dev_params[6] * X**2 * Y + 

            # self.dev_params[7] * X**2 * Y ** 2

            # Add C12, C21, C22 terms here if required by your specific model definition
        )
        
        # --- Volumetric Terms (U) ---
        # Assuming vol_params = [d0, d1] are D2 and D1 parameters (inverse bulk moduli)
        J = jnp.sqrt(i3)
        J_minus_1 = J - 1.0

        # Assuming the volumetric function U(J) = (1/D1)(J-1)^2 + (1/D2)(J-1)^4
        # with D1=d1 and D2=d0 (or vice versa, depending on convention)
        
        # D1 is typically the lower order term (quadratic, hence d1)
        # D2 is typically the higher order term (quartic, hence d0)
        vol_terms = (
            # (1/D1) * (J - 1)**2
            (self.vol_param) * J_minus_1**2 
        #     # (1/D2) * (J - 1)**4
        #     (self.vol_params[1]) * J_minus_1**4
        )
        
        return dev_terms + vol_terms

In [36]:
import numpy as np
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import GridSearchCV
def denoise(coords, u) :
    # 1. Prepare your data
    # coords: array of shape (n_nodes, 2) -> [[x1, y1], [x2, y2], ...]
    # noisy_u: array of shape (n_nodes, 2) -> [[ux1, uy1], [ux2, uy2], ...]
    X = coords[:, :2] 
    y = u

    # 2. Define Hyperparameters
    # alpha: Regularization strength (the "Ridge"). Higher = smoother/more noise filtered.
    # gamma: Kernel bandwidth (for RBF). Controls how "local" the smoothing is.
    # Using GridSearchCV is recommended to find the best balance automatically.
    param_grid = {
        "alpha": [1e0, 0.1, 1e-2, 1e-3],
        "gamma": np.logspace(-2, 2, 5)
    }

    # 3. Initialize and Fit KRR
    # We use the RBF (Radial Basis Function) kernel for smooth displacement fields.
    krr_model = GridSearchCV(
        KernelRidge(kernel="rbf"), 
        param_grid=param_grid, 
        cv=5
    )

    krr_model.fit(X, y)

    # 4. Extract Denoised Data
    # You can predict on the same nodes or a new, coarser mesh.
    denoised_u = krr_model.predict(X)
    return denoised_u

In [37]:
# base_save_path = "saved_model"  # change as needed
# os.makedirs(base_save_path, exist_ok=True)

# # Subfolder with datetime
# timestamp = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
# save_path = os.path.join(base_save_path, timestamp)
# os.makedirs(save_path, exist_ok=True)
material_model = "mooneyrivlin_disp_controlled"
dataset = TractionDataset("dataset",material_model)
F_all = []
u_list = []
load_parameters = []
reaction_list = []
n_t = len(dataset)
for loadstep in range(0, n_t) :
    # if (loadstep != n_t - 1) and (loadstep != 0) and (load) :
    #     continue 
    data = dataset[loadstep]
    coords = data["mesh_pos"][:,:2]
    cells = data["cells"]
    # u = data["u"]
    percent_noise = 0.0001
    node_type = data["node_type"]
    ux = data["u"][:, 0]
    ux[(data["node_type"] != 1)] += np.random.normal(0, percent_noise * 1, ux.shape)[(data["node_type"] != 1)]
    ux[(data["node_type"] != 3)] += np.random.normal(0, percent_noise * 1, ux.shape)[(data["node_type"] != 3)]
    uy = data["u"][:, 1]
    uy[(data["node_type"] != 2)] += np.random.normal(0, percent_noise * 1, uy.shape)[(data["node_type"] != 2)]
    uy[(data["node_type"] != 4)] += np.random.normal(0, percent_noise * 1, uy.shape)[(data["node_type"] != 4)]

    # Combine components into the full displacement vector u
    u_noisy = np.column_stack((ux, uy))
    # u[node_type == 0] = u[node_type == 0] + jax.random.normal(jr.key(0), u.shape)[node_type == 0] * 0.01 * mean_u
    load_parameter = data["load_parameter"]
    u = denoise(coords, u_noisy)
    coord_cells = coords[cells]
    u_cells = u[cells]

    F, dNdx = deformation_gradient_element(coord_cells, u_cells)
    load_parameters.append(load_parameter)
    u_list.append(u)
    F_all.append(F)
    reaction_list.append(data["reaction"])
    # reaction_list.append(jnp.array([0, 0, 0, 0]))
# I_obs_all,_ = invariants_and_derivatives_vmap(F_all_stacked)
u_list = jnp.array(u_list)
load_parameters = jnp.array(load_parameters)
reaction_list = jnp.array(reaction_list)
I_obs, _ = jax.vmap(invariants_and_derivatives)(F)

In [32]:
reaction_list

Array([[-0.48796, -0.55994,  0.48796,  0.55994],
       [-1.09141, -1.19577,  1.09141,  1.19577],
       [-1.81691, -1.91888,  1.81691,  1.91888],
       [-2.66596, -2.72917,  2.66596,  2.72917],
       [-3.63299, -3.62004,  3.63299,  3.62004],
       [-4.70373, -4.58   ,  4.70373,  4.58   ],
       [-5.85482, -5.59394,  5.85482,  5.59394],
       [-7.05584, -6.64501,  7.05584,  6.64501],
       [-8.27388, -7.71727,  8.27388,  7.71727],
       [-9.47885, -8.79789,  9.47885,  8.79789]], dtype=float64)

In [38]:
def _neumann_cell_force(coords_el, types_el, t3, t4):
    """
    coords_el: (3,2) float - coordinates of the 3 nodes of the element
    types_el:  (3,) int - node_type for these 3 nodes (global node_type[cells])
    t3, t4: scalars - traction magnitudes for types 3 and 4
    returns: (3,2) local nodal traction vector for this element
    """
    edges = jnp.array([[0, 1],
                       [1, 2],
                       [2, 0]])  # three local edges
    f_cell = jnp.zeros((3, 2))

    def body_fun(idx, f):
        i = edges[idx, 0]
        j = edges[idx, 1]

        ti = types_el[i]
        tj = types_el[j]

        # Only apply traction if both nodes of the edge have the same neumann type.
        is_right = (ti == 3) & (tj == 3)
        is_top   = (ti == 4) & (tj == 4)

        # choose traction vector for edge
        t_edge = jnp.where(is_right, jnp.array([t3, 0.0]),
                 jnp.where(is_top,   jnp.array([0.0, t4]),
                                         jnp.array([0.0, 0.0])))

        xi = coords_el[i]
        xj = coords_el[j]
        L = jnp.linalg.norm(xj - xi)

        # nodal contribution from this edge: each edge contributes L/2 * T to each of its two nodes
        fe_local = 0.5 * L * t_edge  # shape (2,)

        f = f.at[i].add(fe_local)
        f = f.at[j].add(fe_local)
        return f

    f_cell = jax.lax.fori_loop(0, 3, body_fun, f_cell)
    return f_cell  # (3,2)

def total_physical_loss(n_t, params, coords, cells, u_list, n_nodes, node_type, load_parameters, reactions) :

    total_loss = jnp.sum(jax.vmap(physical_loss, in_axes = (None, None, None, 0, None, None, 0, 0))(params, coords, cells, u_list, n_nodes, node_type, load_parameters, reactions))

    lasso_penalty = jnp.sum((jnp.exp(params))**2) * 0.0
    return total_loss + lasso_penalty
def physical_loss(params, coords, cells, u,
                  n_nodes, node_type, load_parameter, reactions):
    """
    Virtual Field Method Weak form loss
    params: Hyperparameter of Gaussian Process
    coords: (C, 3, 2) per-element nodal coords
    cells:  (C, 3) global node indices per element
    u: displacement (format compatible with deformation_gradient_element)
    reaction_forces: target reaction vector (4,) or similar used previously
    n_nodes: total number of nodes
    bc: (n_nodes, 2) boundary code mask (0 free, 1..4 etc)
    node_type: (n_nodes, 1) ints: 0 free, 1/2 fixed (dirichlet), 3 right, 4 top
    load_parameter: (2,) or (2,1) - [t3, t4]
    """
    u_cells = u[cells]
    coord_cells = coords[cells]
    # --- INTERNAL FORCES (unchanged) ---
    F, dNdx = deformation_gradient_element(coord_cells, u_cells)   # (C,2,2), (C,3,2,2?) matches your API
    dA = jnp.linalg.det(transformation_jacobian(coord_cells)) / 2  # (C,)

    # hyperGP = SparseHyperelasticityGP(params, Z_I)
    f = jax.vmap(fto3x3)(F)
    # piola = jax.vmap(lambda x: hyperGP.piola_stress(x, key))(f)[:, :2, :2]
    material_model = MooneyRivlin(jnp.exp(params))
    piola = jax.vmap(material_model.P)(f)[:, :2, :2]
    # internal element nodal forces: (C,3,2)
    f_int_cell = jnp.einsum("cij, cnj -> cin", piola, dNdx) * dA[:, None, None]
    f_int_cell = jnp.swapaxes(f_int_cell, 1, 2)    # (C,3,2)

    # assemble into global internal force vector (n_nodes, 2)
    f_int_nodes = jnp.zeros((n_nodes, 2)).at[cells].add(f_int_cell)
    
    # --- NEUMANN EDGE-LENGTH TRACTION ---
    # normalize load_parameter to flat array
    t3 = load_parameter * 0.99 # TO ADD ADJUSTABLE load_parameters
    t4 = load_parameter  # TO ADD ADJUSTABLE load_parameters
    # node_type may be (n_nodes,1) so flatten
    node_type_flat = jnp.asarray(node_type).reshape(-1)  # (n_nodes,)
    types_per_cell = node_type_flat[cells]               # (C,3)

    # vectorize per-element traction computation
    per_cell_vmap = jax.vmap(_neumann_cell_force, in_axes=(0, 0, None, None))
    f_neu_cells = per_cell_vmap(coord_cells, types_per_cell, t3, t4)  # (C,3,2)

    # assemble global neumann nodal forces
    f_neu_nodes = jnp.zeros((n_nodes, 2)).at[cells].add(f_neu_cells)

    # --- Residual R = int(grad v : P) dx  -  int(v·T) ds(Neumann)
    # R_nodes = f_int_nodes - f_neu_nodes
    # r = jnp.sum(R_nodes * u, axis = -1)
    # r_total = jnp.sum(r**2)
    # node_type = 1 means (c, i) on DBC with i = 0
    # node_type = 2 means (c, i) on DBC with i = 1
    # node_type = 3 means (c, i) on DBC with i = 0
    # node_type = 4 means (c, i) on DBC with i = 1 
    free_r0 = jnp.sum(f_int_nodes[node_type == 0] ** 2)
    free_r1 = jnp.sum(f_int_nodes[node_type == 1, 1] ** 2)
    free_r2 = jnp.sum(f_int_nodes[node_type == 2, 0] ** 2)
    free_r3 = jnp.sum(f_int_nodes[node_type == 3, 1] ** 2)
    free_r4 = jnp.sum(f_int_nodes[node_type == 4, 0] ** 2)
    free_r_total = free_r0 + free_r1 + free_r2 + free_r3 + free_r4

    # # only free DOFs contribute to the residual loss (bc == 0)
    # blm_loss = jnp.sum(R_nodes[(node_type != 1) & (node_type != 2)] ** 2)
    # # loss at the dirichlet nodes
    # fixed_nodes_loss1 = jnp.sum((jnp.sum(f_int_nodes[node_type == 1], axis = 0) + jnp.sum(f_neu_nodes[node_type == 3], axis = 0))**2)
    # fixed_nodes_loss2 = jnp.sum((jnp.sum(f_int_nodes[node_type == 2], axis = 0) + jnp.sum(f_neu_nodes[node_type == 4], axis = 0))**2)
    
    fnl_left = (jnp.sum(f_int_nodes[node_type == 1, 0]) - reactions[0])**2
    fnl_bottom = ((jnp.sum(f_int_nodes[node_type == 2, 1]) - reactions[1])**2)
    fnl_right = ((jnp.sum(f_int_nodes[node_type == 3, 0]) - reactions[2])**2)
    fnl_top = ((jnp.sum(f_int_nodes[node_type == 4, 1]) - reactions[3])**2)

    reaction_loss = fnl_left + fnl_bottom + fnl_right + fnl_top
    # # total_physic_loss = blm_loss
    # # lasso_penalty = 0.1 * jnp.sum(jnp.abs(jnp.exp(params)))
    # total_physic_loss = blm_loss + fixed_nodes_loss1 + fixed_nodes_loss2

    # free_loss = jnp.sum(f_int_nodes[(node_type != 1) & (node_type != 2)]**2)

    # # 2. Boundary Condition Match on Neumann Nodes
    # # (Ensures the stress at the pulling end matches the traction you applied)
    # neumann_loss_x = jnp.sum((f_int_nodes[node_type == 3] - f_neu_nodes[node_type == 3])**2)
    # neumann_loss_y = jnp.sum((f_int_nodes[node_type == 4] + f_neu_nodes[node_type == 4])**2)

    # # 3. Global Reaction Balance at Dirichlet Nodes
    # # (Ensures the 'machine' reads the correct total force)
    # total_applied_force_x = jnp.sum(f_neu_nodes[node_type == 3], axis=0)
    # total_applied_force_y = jnp.sum(f_neu_nodes[node_type ==4 ], axis=0)
    # # total_internal_reaction = jnp.sum(f_int_nodes[node_type == 1], axis=0)

    # internal_reaction_x = jnp.sum(f_int_nodes[node_type == 1], axis=0)
    # internal_reaction_y = jnp.sum(f_int_nodes[node_type == 2], axis=0)

    # global_loss = jnp.sum((internal_reaction_x + internal_reaction_y + total_applied_force_y + total_applied_force_x)**2)

    # total_physic_loss = blm_loss + global_loss
    # reaction_loss = fixed_nodes_loss1 + fixed_nodes_loss2
    return free_r_total  + reaction_loss

In [28]:
import jaxopt

In [22]:
import optax
from tqdm import tqdm
# trainable parameters
params = jnp.zeros(6) 

# choose optimizer
# opt = optax.lbfgs(1e-3)
opt = optax.adam(5e-3)
opt_state = opt.init(params)

loss_and_grad = jax.jit(jax.value_and_grad(
    lambda params: total_physical_loss(n_t, params, coords, cells, u_list, coords.shape[0], node_type, load_parameters, reaction_list)
))
bar = tqdm(range(50000))
for step in tqdm(bar):
    loss, grads = loss_and_grad(params)
    updates, opt_state = opt.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

    bar.set_postfix({"step": f"{step:04d}", "loss":f"{loss:.6f}", "params": f"{jnp.exp(params)}"})
        # print(f"step {step:04d}  loss={loss:.6f} {jnp.exp(params)}")



  0%|          | 0/50000 [00:00<?, ?it/s]

(3, 3)


  0%|          | 104/50000 [00:02<21:54, 37.95it/s]


KeyboardInterrupt: 

In [39]:
import optax
from tqdm import tqdm
# trainable parameters
params = jnp.zeros(6) 
def loss_fn(params):
    return total_physical_loss(
        n_t, params, coords, cells, u_list, 
        coords.shape[0], node_type, load_parameters, reaction_list
    )
solver = jaxopt.LBFGS(fun=loss_fn, maxiter=5000, verbose = True)

# 3. Use jax.lax.scan for the inner loop (if doing multiple restarts/steps)
# However, for L-BFGS, usually one call to .run() is enough
res = solver.run(params)
params = res.params
print(f"Final loss: {res.state.error}")

(3, 3)
(3, 3)
(3, 3)
(3, 3)
(3, 3)
(3, 3)
(3, 3)
(3, 3)
INFO: jaxopt.ZoomLineSearch: Iter: 1, Stepsize: 1.0, Decrease error: 9.141719233391829e+89, Curvature error: 1.8399779933708133e+92
INFO: jaxopt.ZoomLineSearch: Iter: 2, Stepsize: 0.5, Decrease error: 1.7998185843658276e+46, Curvature error: 3.6225424373093866e+48
INFO: jaxopt.ZoomLineSearch: Iter: 3, Stepsize: 0.33333333333333337, Decrease error: 4.859918835401323e+31, Curvature error: 9.781687096715841e+33
INFO: jaxopt.ZoomLineSearch: Iter: 4, Stepsize: 0.22222222222222154, Decrease error: 9.423868243131454e+21, Curvature error: 1.8967668709699605e+24
INFO: jaxopt.ZoomLineSearch: Iter: 5, Stepsize: 0.14814814811582938, Decrease error: 3157152012534900.0, Curvature error: 6.354486983351973e+17
INFO: jaxopt.ZoomLineSearch: Iter: 6, Stepsize: 0.0987653948531282, Decrease error: 152264426080.75467, Curvature error: 30649205977577.117
INFO: jaxopt.ZoomLineSearch: Iter: 7, Stepsize: 0.06584002370070603, Decrease error: 200743516.06300

In [40]:
jnp.exp(params)

Array([0.     , 1.10644, 0.35656, 0.     , 0.     , 1.56431], dtype=float64)

In [53]:
r = 1e-10
true_params = jnp.zeros(9)
true_params = true_params.at[0].add(jnp.log(0.5))
true_params = true_params.at[1].add(jnp.log(1.0))
true_params = true_params.at[2].add(jnp.log(1.0))
true_params = true_params.at[3].add(jnp.log(r))
true_params = true_params.at[4].add(jnp.log(r))
# true_params = true_params.at[1].add(jnp.log(r))
# true_params = true_params.at[2].add(jnp.log(r))
# true_params = true_params.at[3].add(jnp.log(r))
# true_params = true_params.at[4].add(jnp.log(r))
true_params = true_params.at[-1].add(jnp.log(1.50)) 

In [54]:
loss, grads = loss_and_grad(true_params)

(3, 3)
(3, 3)
(3, 3)
(3, 3)
(3, 3)
(3, 3)
(3, 3)
(3, 3)
(3, 3)
(3, 3)


In [56]:
loss_and_grad(true_params)

(Array(0.24843, dtype=float64),
 Array([ -0.49217,  -1.0543 ,  -1.28301,  -0.     ,  -0.     ,   0.     ,   0.     ,   0.     , -13.18251], dtype=float64))

In [55]:
true_params

Array([ -0.69315,   0.     ,   0.     , -23.02585, -23.02585,   0.     ,   0.     ,   0.     ,   0.40547], dtype=float64)

In [193]:
jnp.array([[1,2],[2,3]]) * jnp.array([[1,2],[2,3]])

Array([[1, 4],
       [4, 9]], dtype=int64)

In [96]:
loss_and_grad = jax.jit(jax.value_and_grad(
    lambda params: physical_loss(params, coord_cells, cells, u_cells, coords.shape[0], node_type, load_parameter)
))

In [97]:
loss_and_grad(true_params)

(3, 3)


(Array(0.13987, dtype=float64),
 Array([ 0.44538,  0.0609 ,  1.10524,  0.     ,  0.     ,  0.     ,  0.     ,  0.     , -4.42951], dtype=float64))